### Mask Strategies

Shear-Shear: ell max 4000
    1. first zero point of sphyrical harmonic function (probe dependent; redshift independent)
    2. limber approximation (redshift dependent)
Shear-Galaxy: k max 0.3
    consider both:
    1. minimum transverse radius (redshift dependent)
    2. ell max 4000
Galaxy-Galaxy: k max 0.3
    1. minimum transverse radius (redshift dependent)

In [1]:
import numpy as np

vmin = 1
vmax = 500
N = 20
logdt = (np.log(vmax) - np.log(vmin))/N
fac = (2./3.)

thetas = []
for i in range(N):
    thetamin = np.exp(np.log(vmin) + (i + 0.)*logdt)
    thetamax = np.exp(np.log(vmin) + (i + 1.)*logdt)
    thetas.append(fac * (thetamax**3 - thetamin**3) / (thetamax**2 - thetamin**2))

In [2]:
import glob
nsrcs = 8
files_p = glob.glob('./files4scalecut/thetacuts_p*')
files_p = np.sort(files_p)
thetacuts_p = np.ones((nsrcs,nsrcs))*-1
cnt = 0
for i in range(nsrcs):
    for j in range(i,nsrcs):
        thetacuts_p[i,j] = np.load(files_p[cnt])
        cnt+=1 
files_m = glob.glob('./files4scalecut/thetacuts_m*')
files_m = np.sort(files_m)
thetacuts_m = np.ones((nsrcs,nsrcs))*-1
cnt = 0
for i in range(nsrcs):
    for j in range(i,nsrcs):
        thetacuts_m[i,j] = np.load(files_m[cnt])
        cnt+=1 

In [3]:
thetas = np.array(thetas)
nsrcs = 8
nlens = 10
lmax = 3000

choice_shear = 3

if choice_shear == 1:
    # hankel first zero point
    from scipy import special
    j0 = special.jn_zeros(0,1)[0]
    j4 = special.jn_zeros(4,1)[0]
    xi_plus_min = j0/lmax*180/np.pi*60
    xi_minus_min = j4/lmax*180/np.pi*60
    print(f'xi_plus_min={xi_plus_min}, xi_minus_min={xi_minus_min}')
elif choice_shear == 2:
    # pi/ell
    xi_plus_min = 3.6
    xi_minus_min = 3.6
elif choice_shear == 3:
    # kmode scalecut
    xi_plus_min = 1
    xi_minus_min = 5
elif choice_shear == 0:
    # no scale cut
    xi_plus_min = 0.
    xi_minus_min = 0.

choice_lens = 1
if choice_lens==0:
    # DES scenario
    # Rmin=6Mpc/h, 8Mpc/h
    gammat_min = [28.58449298905984, 21.36042614498998, 17.188966845559126, 14.499280303830677, 12.630847712055141, 11.261848411483253, 10.217945955405659, 9.396911952703803, 8.735005165892007, 8.190496296627623]
    wtheta_min = [38.11265731874644, 28.48056819331998, 22.918622460745503, 19.332373738440907, 16.84113028274019, 15.015797881977669, 13.623927940540879, 12.529215936938403, 11.646673554522673, 10.920661728836832]
elif choice_lens==1:
    # LSST scenario
    # Rmin=21Mpc/h
    gammat_min = [99.77870353446592, 74.56195317183098, 60.000813294230085, 50.61203609428183, 44.08997597867098, 39.311266928175186, 35.66735994253329, 32.801410618938206, 30.4909200647058, 28.590225549710834]
    wtheta_min = [99.77870353446592, 74.56195317183098, 60.000813294230085, 50.61203609428183, 44.08997597867098, 39.311266928175186, 35.66735994253329, 32.801410618938206, 30.4909200647058, 28.590225549710834]
ggl_exclude = []
Ntheta = 20
Ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens-len(ggl_exclude))*Ntheta
mask = np.ones(Ndv)

ncombo = 0
ncombo0 = 0
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if choice_shear==3:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>thetacuts_p[i,j])
        else:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_plus_min)
        print(f'xi+: ni={i}, nj={j}, cut {np.sum(thetas<xi_plus_min)} points')
        ncombo += 1
ncombo1 = ncombo
print(f'xi+ completed, {np.sum(mask[ncombo0*Ntheta: ncombo1*Ntheta])}/{len(mask[ncombo0*Ntheta: ncombo1*Ntheta])} left')
print('xi+ completed, index now is ', ncombo)
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if choice_shear==3:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>thetacuts_m[i,j])
        else:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_minus_min)
        print(f'xi-: ni={i}, nj={j}, cut {np.sum(thetas<xi_minus_min)} points')
        ncombo += 1
ncombo2 = ncombo
print(f'xi- completed, {np.sum(mask[ncombo1*Ntheta: ncombo2*Ntheta])}/{len(mask[ncombo1*Ntheta: ncombo2*Ntheta])} left')
print('xi- completed, index now is ', ncombo)
for i in range(nlens):
    for j in range(nsrcs):
        if (i,j) in ggl_exclude:
            continue
        mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>gammat_min[i])
        print(f'gammat: ni={i}, nj={j}, cut {np.sum(thetas<gammat_min[i])} points')
        ncombo += 1
ncombo3 = ncombo
print(f'gammat completed, {np.sum(mask[ncombo2*Ntheta: ncombo3*Ntheta])}/{len(mask[ncombo2*Ntheta: ncombo3*Ntheta])} left')
print('gammat completed, index now is ', ncombo)
for i in range(nlens):
    mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>wtheta_min[i])
    print(f'gammat: ni={i}, nj={j}, cut {np.sum(thetas<wtheta_min[i])} points')
    ncombo += 1
ncombo4 = ncombo
print(f'gammat completed, {np.sum(mask[ncombo3*Ntheta: ncombo4*Ntheta])}/{len(mask[ncombo3*Ntheta: ncombo4*Ntheta])} left')
print('wtheta completed, index now is ', ncombo)

xi+: ni=0, nj=0, cut 0 points
xi+: ni=0, nj=1, cut 0 points
xi+: ni=0, nj=2, cut 0 points
xi+: ni=0, nj=3, cut 0 points
xi+: ni=0, nj=4, cut 0 points
xi+: ni=0, nj=5, cut 0 points
xi+: ni=0, nj=6, cut 0 points
xi+: ni=0, nj=7, cut 0 points
xi+: ni=1, nj=1, cut 0 points
xi+: ni=1, nj=2, cut 0 points
xi+: ni=1, nj=3, cut 0 points
xi+: ni=1, nj=4, cut 0 points
xi+: ni=1, nj=5, cut 0 points
xi+: ni=1, nj=6, cut 0 points
xi+: ni=1, nj=7, cut 0 points
xi+: ni=2, nj=2, cut 0 points
xi+: ni=2, nj=3, cut 0 points
xi+: ni=2, nj=4, cut 0 points
xi+: ni=2, nj=5, cut 0 points
xi+: ni=2, nj=6, cut 0 points
xi+: ni=2, nj=7, cut 0 points
xi+: ni=3, nj=3, cut 0 points
xi+: ni=3, nj=4, cut 0 points
xi+: ni=3, nj=5, cut 0 points
xi+: ni=3, nj=6, cut 0 points
xi+: ni=3, nj=7, cut 0 points
xi+: ni=4, nj=4, cut 0 points
xi+: ni=4, nj=5, cut 0 points
xi+: ni=4, nj=6, cut 0 points
xi+: ni=4, nj=7, cut 0 points
xi+: ni=5, nj=5, cut 0 points
xi+: ni=5, nj=6, cut 0 points
xi+: ni=5, nj=7, cut 0 points
xi+: ni=6,

### this is for saving directly use mask

In [4]:
print('total number of datavecotr is ', (Ndv/20-len(ggl_exclude))*20)
print('number of data vector after scale cut is ', np.sum(mask))

nlen = len(mask)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask
np.savetxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/roman_SRD.mask', out)

total number of datavecotr is  3240.0
number of data vector after scale cut is  1946.0


### this is for saving mask need further chi2 process 

In [9]:
nlen = len(mask)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask
np.savetxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/roman_full_shear_DESY3_lens.mask', out)

### this is for lens=source

In [5]:
import numpy as np

vmin = 1
vmax = 500
N = 20
logdt = (np.log(vmax) - np.log(vmin))/N
fac = (2./3.)

thetas = []
for i in range(N):
    thetamin = np.exp(np.log(vmin) + (i + 0.)*logdt)
    thetamax = np.exp(np.log(vmin) + (i + 1.)*logdt)
    thetas.append(fac * (thetamax**3 - thetamin**3) / (thetamax**2 - thetamin**2))

thetas = np.array(thetas)
nsrcs = 8
nlens = 8
lmax = 3000

choice_shear = 0

if choice_shear == 1:
    # hankel first zero point
    from scipy import special
    j0 = special.jn_zeros(0,1)[0]
    j4 = special.jn_zeros(4,1)[0]
    xi_plus_min = j0/lmax*180/np.pi*60
    xi_minus_min = j4/lmax*180/np.pi*60
    print(f'xi_plus_min={xi_plus_min}, xi_minus_min={xi_minus_min}')
elif choice_shear == 2:
    # pi/ell
    xi_plus_min = 3.6
    xi_minus_min = 3.6
elif choice_shear == 3:
    # kmode scalecut
    xi_plus_min = 1
    xi_minus_min = 5
elif choice_shear == 0:
    # no scale cut
    xi_plus_min = 0.
    xi_minus_min = 0.

choice_lens = 0
if choice_lens==0:
    # DES scenario
    # Rmin=6Mpc/h, 8Mpc/h
    gammat_min = [23.879658951415006, 14.578470322032384, 11.299264368485323, 9.370908663586293, 8.022872965372477, 6.963104229248865, 6.010663060951033, 5.007883180682353]
    wtheta_min = [31.83954526855334, 19.437960429376513, 15.065685824647096, 12.494544884781725, 10.697163953829968, 9.28413897233182, 8.014217414601378, 6.677177574243137]
elif choice_lens==1:
    # LSST scenario
    # Rmin=21Mpc/h
    gammat_min = [99.77870353446592, 74.56195317183098, 60.000813294230085, 50.61203609428183, 44.08997597867098, 39.311266928175186, 35.66735994253329, 32.801410618938206, 30.4909200647058, 28.590225549710834]
    wtheta_min = [99.77870353446592, 74.56195317183098, 60.000813294230085, 50.61203609428183, 44.08997597867098, 39.311266928175186, 35.66735994253329, 32.801410618938206, 30.4909200647058, 28.590225549710834]
ggl_exclude = [(6, 0), (7, 0), (7, 1)]
Ntheta = 20
Ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens-len(ggl_exclude))*Ntheta
mask = np.ones(Ndv)

ncombo = 0
ncombo0 = 0
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if choice_shear==3:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>thetacuts_p[i,j])
        else:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_plus_min)
        print(f'xi+: ni={i}, nj={j}, cut {np.sum(thetas<xi_plus_min)} points')
        ncombo += 1
ncombo1 = ncombo
print(f'xi+ completed, {np.sum(mask[ncombo0*Ntheta: ncombo1*Ntheta])}/{len(mask[ncombo0*Ntheta: ncombo1*Ntheta])} left')
print('xi+ completed, index now is ', ncombo)
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if choice_shear==3:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>thetacuts_m[i,j])
        else:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_minus_min)
        print(f'xi-: ni={i}, nj={j}, cut {np.sum(thetas<xi_minus_min)} points')
        ncombo += 1
ncombo2 = ncombo
print(f'xi- completed, {np.sum(mask[ncombo1*Ntheta: ncombo2*Ntheta])}/{len(mask[ncombo1*Ntheta: ncombo2*Ntheta])} left')
print('xi- completed, index now is ', ncombo)
for i in range(nlens):
    for j in range(nsrcs):
        if (i,j) in ggl_exclude:
            continue
        mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>gammat_min[i])
        print(f'gammat: ni={i}, nj={j}, cut {np.sum(thetas<gammat_min[i])} points')
        ncombo += 1
ncombo3 = ncombo
print(f'gammat completed, {np.sum(mask[ncombo2*Ntheta: ncombo3*Ntheta])}/{len(mask[ncombo2*Ntheta: ncombo3*Ntheta])} left')
print('gammat completed, index now is ', ncombo)
for i in range(nlens):
    mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>wtheta_min[i])
    print(f'wtheta: ni={i}, nj={i}, cut {np.sum(thetas<wtheta_min[i])} points')
    ncombo += 1
ncombo4 = ncombo
print(f'wtheta completed, {np.sum(mask[ncombo3*Ntheta: ncombo4*Ntheta])}/{len(mask[ncombo3*Ntheta: ncombo4*Ntheta])} left')
print('wtheta completed, index now is ', ncombo)

xi+: ni=0, nj=0, cut 0 points
xi+: ni=0, nj=1, cut 0 points
xi+: ni=0, nj=2, cut 0 points
xi+: ni=0, nj=3, cut 0 points
xi+: ni=0, nj=4, cut 0 points
xi+: ni=0, nj=5, cut 0 points
xi+: ni=0, nj=6, cut 0 points
xi+: ni=0, nj=7, cut 0 points
xi+: ni=1, nj=1, cut 0 points
xi+: ni=1, nj=2, cut 0 points
xi+: ni=1, nj=3, cut 0 points
xi+: ni=1, nj=4, cut 0 points
xi+: ni=1, nj=5, cut 0 points
xi+: ni=1, nj=6, cut 0 points
xi+: ni=1, nj=7, cut 0 points
xi+: ni=2, nj=2, cut 0 points
xi+: ni=2, nj=3, cut 0 points
xi+: ni=2, nj=4, cut 0 points
xi+: ni=2, nj=5, cut 0 points
xi+: ni=2, nj=6, cut 0 points
xi+: ni=2, nj=7, cut 0 points
xi+: ni=3, nj=3, cut 0 points
xi+: ni=3, nj=4, cut 0 points
xi+: ni=3, nj=5, cut 0 points
xi+: ni=3, nj=6, cut 0 points
xi+: ni=3, nj=7, cut 0 points
xi+: ni=4, nj=4, cut 0 points
xi+: ni=4, nj=5, cut 0 points
xi+: ni=4, nj=6, cut 0 points
xi+: ni=4, nj=7, cut 0 points
xi+: ni=5, nj=5, cut 0 points
xi+: ni=5, nj=6, cut 0 points
xi+: ni=5, nj=7, cut 0 points
xi+: ni=6,

In [6]:
nlen = len(mask)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask
np.savetxt('./data/roman_les_full_shear_DESY3_lens.mask', out)